# Feature Engineering - DOGE/USDT Technical Indicators

El objetivo de este notebook es transformar el dataset bruto de mercado en un conjunto de datos enriquecido mediante técnicas de feature engineering orientadas al análisis financiero y al trading algorítmico.

Para ello, se generan indicadores técnicos, métricas de volatilidad, retornos, variables derivadas y etiquetas supervisadas que permitirán entrenar y evaluar modelos predictivos capaces de identificar patrones de comportamiento en el mercado de criptomonedas.

Los gaps detectados en la frecuencia esperada de 5 minutos no se rellenan con datos sintéticos; se utilizan para dividir la serie en segmentos continuos y evitar que retornos, ventanas móviles e indicadores crucen saltos temporales.

In [1]:
# ============================================================
# Imports and load dataset
# Carga el dataset bruto de velas de DOGE/USDT, convierte las columnas temporales a formato datetime, ordena los datos cronológicamente y prepara la ruta de salida para guardar el dataset enriquecido.
# ============================================================

import os
import numpy as np
import pandas as pd

RAW_PATH = "../data/raw/DOGEUSDT_5m_binance_2017_2026.csv"
OUTPUT_DIR = "../data/processed"
OUTPUT_FILE = "DOGEUSDT_5m_binance_2017_2026_features.csv"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(RAW_PATH, parse_dates=["open_time", "close_time"])
df = df.sort_values("open_time").reset_index(drop=True)

print("Loaded raw dataset")
print("Shape:", df.shape)
print("Date range:", df["open_time"].min(), "->", df["open_time"].max())

display(df.head())

Loaded raw dataset
Shape: (723409, 11)
Date range: 2019-07-05 12:00:00 -> 2026-05-23 11:10:00


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
0,2019-07-05 12:00:00,0.004490,0.004600,0.003760,0.004350,281690734.0,2019-07-05 12:04:59.999,1.213789e+06,1841,131946467.0,572788.485474
1,2019-07-05 12:05:00,0.004340,0.004340,0.004000,0.004022,151285521.0,2019-07-05 12:09:59.999,6.310450e+05,963,41885000.0,173783.101331
2,2019-07-05 12:10:00,0.004022,0.004069,0.003800,0.003870,182657540.0,2019-07-05 12:14:59.999,7.079442e+05,1285,58720314.0,228693.805868
3,2019-07-05 12:15:00,0.003861,0.003940,0.003859,0.003929,73986015.0,2019-07-05 12:19:59.999,2.885708e+05,535,43864831.0,171156.394506
4,2019-07-05 12:20:00,0.003929,0.003964,0.003870,0.003877,67030342.0,2019-07-05 12:24:59.999,2.627509e+05,399,27619614.0,108776.180726


El dataset queda correctamente cargado y ordenado de forma cronológica, con más de 700.000 velas de 5 minutos entre julio de 2019 y mayo de 2026. Además de validar el rango temporal, se identifican discontinuidades puntuales en la frecuencia esperada.

En lugar de rellenar los huecos con velas sintéticas, el dataset se divide en segmentos temporales continuos. Esta decisión evita introducir precios o volúmenes artificiales y permite calcular indicadores técnicos, retornos y etiquetas futuras únicamente dentro de tramos donde la continuidad temporal está garantizada.

In [2]:
# ============================================================
# Temporal continuity validation
# Detecta gaps respecto a la frecuencia esperada de 5 minutos y crea segmentos continuos para evitar que features y targets crucen discontinuidades temporales.
# ============================================================

EXPECTED_FREQ = pd.Timedelta(minutes=5)

df["time_diff"] = df["open_time"].diff()
df["gap_flag"] = df["time_diff"].ne(EXPECTED_FREQ)

# The first row is not a real gap; it only has no previous timestamp.
df.loc[df.index[0], "gap_flag"] = False

detected_gaps = df[df["gap_flag"]].copy()

print("Expected candle frequency:", EXPECTED_FREQ)
print("Detected temporal gaps:", len(detected_gaps))

if len(detected_gaps) > 0:
    display(
        detected_gaps[
            ["open_time", "time_diff", "open", "high", "low", "close", "volume"]
        ].head(20)
    )

# Each segment contains consecutive 5-minute candles.
df["segment_id"] = df["gap_flag"].cumsum()

print("Continuous temporal segments:", df["segment_id"].nunique())
print("Segment size summary:")
display(df.groupby("segment_id").size().describe())

Expected candle frequency: 0 days 00:05:00
Detected temporal gaps: 18


,open_time,time_diff,open,high,low,close,volume
11688,2019-08-15 10:00:00,0 days 08:05:00,0.002684,0.002684,0.002619,0.002637,5177724.0
37512,2019-11-13 04:20:00,0 days 02:25:00,0.002658,0.002658,0.002658,0.002658,0.0
40940,2019-11-25 04:00:00,0 days 02:05:00,0.002189,0.002189,0.002165,0.002165,1347492.0
62804,2020-02-09 03:00:00,0 days 01:05:00,0.003246,0.003291,0.003233,0.003265,18708621.0
65788,2020-02-19 17:30:00,0 days 05:55:00,0.002813,0.002824,0.002813,0.002819,3571441.0
69723,2020-03-04 11:30:00,0 days 02:10:00,0.002456,0.002456,0.002419,0.002419,3833351.0
84585,2020-04-25 04:30:00,0 days 02:35:00,0.002080,0.002080,0.002080,0.002080,10035.0
102987,2020-06-28 05:30:00,0 days 03:35:00,0.002292,0.002292,0.002292,0.002292,4362.0
147633,2020-11-30 07:00:00,0 days 01:05:00,0.003519,0.003543,0.003515,0.003523,2165239.0
153763,2020-12-21 18:00:00,0 days 04:15:00,0.004567,0.005000,0.004567,0.004906,82218870.0


Continuous temporal segments: 19
Segment size summary:


count        19.000000
mean      38074.157895
std       79253.765481
min         960.000000
25%        5032.500000
50%       13542.000000
75%       23844.000000
max      332895.000000
dtype: float64

La comprobación de continuidad temporal confirma la presencia de 18 gaps respecto a la frecuencia esperada de velas de 5 minutos, coincidiendo con el resultado observado durante la fase de adquisición de datos. Esta consistencia es importante porque confirma que las discontinuidades no se introducen durante el proceso de feature engineering, sino que ya están presentes en el dataset histórico original.

Los 18 gaps detectados dividen el dataset en 19 segmentos temporales continuos. La mayoría de los segmentos contienen un número elevado de velas consecutivas, con el segmento más largo superando las 330.000 observaciones. Por tanto, aunque los gaps son pocos en comparación con el tamaño total del dataset, deben tratarse de forma explícita porque varias operaciones de feature engineering dependen de la continuidad temporal.

Por este motivo, no se completa el dataset artificialmente mediante velas sintéticas. En su lugar, se crea la variable `segment_id`, que permite identificar cada bloque continuo de velas de 5 minutos. Esto permite calcular indicadores con ventanas móviles, medias móviles exponenciales, retornos retardados y targets futuros únicamente dentro de segmentos temporales sin interrupciones.

Este enfoque evita cruzar gaps temporales durante el cálculo de features o etiquetas supervisadas. En términos prácticos, impide que una operación como `shift`, una ventana `rolling` o un target futuro trate dos velas separadas por varias horas como si fueran observaciones consecutivas de 5 minutos. De esta forma, se preserva la coherencia temporal del dataset sin introducir datos de mercado artificiales.

In [3]:
# ============================================================
# Technical indicators
# Esta celda convierte el precio bruto en un conjunto de variables más informativas para un modelo de machine learning.
# El precio por sí solo suele ser pobre como predictor; los indicadores técnicos resumen tendencia, momentum, volatilidad y desviación respecto al comportamiento reciente.
# Los cálculos se realizan por segmento continuo para evitar que retornos, EMAs, ventanas móviles o indicadores crucen gaps temporales.
# ============================================================

# Helper functions for gap-aware calculations

def rolling_by_segment(column, window, aggregation):
    return df.groupby("segment_id")[column].transform(
        lambda s: getattr(s.rolling(window=window), aggregation)()
    )


def ewm_mean_by_segment(column, span):
    return df.groupby("segment_id")[column].transform(
        lambda s: s.ewm(span=span, adjust=False).mean()
    )


# Historical returns (past information only)
previous_close = df.groupby("segment_id")["close"].shift(1)
df["return_prev_1"] = df.groupby("segment_id")["close"].pct_change()
df["log_return_prev_1"] = np.log(df["close"] / previous_close)

# Moving averages
df["sma_20"] = rolling_by_segment("close", 20, "mean")
df["ema_10"] = ewm_mean_by_segment("close", 10)
df["ema_50"] = ewm_mean_by_segment("close", 50)
df["ema_200"] = ewm_mean_by_segment("close", 200)

# EMA ratios
df["ema10_ema50_ratio"] = df["ema_10"] / df["ema_50"]
df["ema50_ema200_ratio"] = df["ema_50"] / df["ema_200"]
df["sma20_ema50_ratio"] = df["sma_20"] / df["ema_50"]

# Rolling volatility: 1 hour = 12 candles of 5 minutes
df["volatility_1h"] = rolling_by_segment("log_return_prev_1", 12, "std")

# Z-score over 1 hour
rolling_mean_1h = rolling_by_segment("close", 12, "mean")
rolling_std_1h = rolling_by_segment("close", 12, "std")
df["zscore_close_1h"] = (df["close"] - rolling_mean_1h) / rolling_std_1h

# RSI 14
delta = df["close"] - previous_close
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.groupby(df["segment_id"]).transform(
    lambda s: s.rolling(window=14).mean()
)
avg_loss = loss.groupby(df["segment_id"]).transform(
    lambda s: s.rolling(window=14).mean()
)

rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

# MACD: 12, 26, 9
ema_12 = ewm_mean_by_segment("close", 12)
ema_26 = ewm_mean_by_segment("close", 26)

df["macd"] = ema_12 - ema_26
df["macd_signal"] = df.groupby("segment_id")["macd"].transform(
    lambda s: s.ewm(span=9, adjust=False).mean()
)
df["macd_hist"] = df["macd"] - df["macd_signal"]

# Bollinger Bands: 20 periods, 2 std
bb_window = 20
bb_std = 2

bb_mid = rolling_by_segment("close", bb_window, "mean")
bb_std_dev = rolling_by_segment("close", bb_window, "std")

df["bb_mid"] = bb_mid
df["bb_upper"] = bb_mid + bb_std * bb_std_dev
df["bb_lower"] = bb_mid - bb_std * bb_std_dev
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["bb_mid"]
df["bb_percent"] = (df["close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])

# ATR 14
high_low = df["high"] - df["low"]
high_close_prev = (df["high"] - previous_close).abs()
low_close_prev = (df["low"] - previous_close).abs()

true_range = pd.concat([high_low, high_close_prev, low_close_prev], axis=1).max(axis=1)
df["atr_14"] = true_range.groupby(df["segment_id"]).transform(
    lambda s: s.rolling(window=14).mean()
)

print("Gap-aware technical features created")
print("Shape:", df.shape)

display(df.tail())

Gap-aware technical features created
Shape: (723409, 35)


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,rsi_14,macd,macd_signal,macd_hist,bb_mid,bb_upper,bb_lower,bb_width,bb_percent,atr_14
723404,2026-05-23 10:50:00,0.09957,0.09961,0.09951,0.09952,2019387.0,2026-05-23 10:54:59.999,201036.76574,672,1267731.0,...,52.873563,-0.000053,-0.000090,0.000038,0.099524,0.099679,0.099368,0.003117,0.488716,0.000115
723405,2026-05-23 10:55:00,0.09952,0.09954,0.09950,0.09952,284994.0,2026-05-23 10:59:59.999,28361.11493,286,211374.0,...,50.602410,-0.000051,-0.000082,0.000031,0.099520,0.099673,0.099368,0.003071,0.498364,0.000110
723406,2026-05-23 11:00:00,0.09952,0.09955,0.09946,0.09948,591135.0,2026-05-23 11:04:59.999,58808.86284,561,257920.0,...,51.219512,-0.000053,-0.000077,0.000024,0.099516,0.099669,0.099364,0.003067,0.380429,0.000110
723407,2026-05-23 11:05:00,0.09948,0.09962,0.09947,0.09954,1837323.0,2026-05-23 11:09:59.999,182909.11863,1442,1026953.0,...,58.536585,-0.000049,-0.000071,0.000022,0.099516,0.099668,0.099364,0.003059,0.578827,0.000114
723408,2026-05-23 11:10:00,0.09954,0.09961,0.09954,0.09956,286859.0,2026-05-23 11:14:59.999,28558.43106,459,182916.0,...,58.536585,-0.000043,-0.000065,0.000022,0.099515,0.099666,0.099364,0.003029,0.649286,0.000111


Tras esta transformación, el dataset pasa de contener únicamente variables de mercado básicas a incorporar indicadores que resumen distintas dimensiones del comportamiento del precio: tendencia, momentum, volatilidad y posición relativa dentro de rangos recientes.

Todas estas variables se calculan usando información pasada o presente de cada vela. Además, al aplicarse por segmentos temporales continuos, se evita que una ventana móvil o un indicador técnico incorpore observaciones separadas por gaps temporales, reduciendo el riesgo de inconsistencias en fases posteriores.

In [4]:
# ============================================================
# Supervised learning labels
# Esta celda define el problema supervisado: dado el estado del mercado en una vela, se intenta predecir si el precio subirá o bajará en distintos horizontes temporales.
# Para el baseline inicial, up_1 o up_3 pueden funcionar como targets sencillos de clasificación binaria.
# Los desplazamientos futuros se calculan por segmento continuo para evitar que los targets crucen gaps temporales.
# ============================================================

horizons = [1, 3, 6, 12]

for h in horizons:
    df[f"future_close_{h}"] = df.groupby("segment_id")["close"].shift(-h)
    df[f"future_return_{h}"] = df[f"future_close_{h}"] / df["close"] - 1
    df[f"up_{h}"] = np.where(
        df[f"future_return_{h}"].notna(),
        (df[f"future_return_{h}"] > 0).astype(int),
        np.nan
    )

print("Gap-aware labels created")
print("Available target columns:")

target_cols = [
    col for col in df.columns
    if col.startswith("future_close_")
    or col.startswith("future_return_")
    or col.startswith("up_")
]
print(target_cols)

# Safety check: old ambiguous target names should not exist anymore
old_target_names = [f"return_{h}" for h in horizons]
old_target_names_found = [col for col in old_target_names if col in df.columns]

if old_target_names_found:
    raise ValueError(
        f"Ambiguous old target columns found: {old_target_names_found}. "
        "Use future_return_* instead."
    )

# Safety check: missing future returns must not be converted into class 0 labels
for h in horizons:
    invalid_labels = df[f"future_return_{h}"].isna() & df[f"up_{h}"].notna()
    if invalid_labels.any():
        raise ValueError(f"Invalid non-null labels found where future_return_{h} is null.")

print("Naming check passed: no ambiguous return_* target columns found.")
print("Null-label safety check passed.")

Gap-aware labels created
Available target columns:
['future_close_1', 'future_return_1', 'up_1', 'future_close_3', 'future_return_3', 'up_3', 'future_close_6', 'future_return_6', 'up_6', 'future_close_12', 'future_return_12', 'up_12']
Naming check passed: no ambiguous return_* target columns found.
Null-label safety check passed.


En esta fase se define explícitamente qué se quiere predecir. Para cada horizonte temporal se crean tanto el retorno futuro como una etiqueta binaria de subida o bajada.

La separación entre return_prev_1, que representa información histórica, y future_return_*, que representa el objetivo futuro, es importante porque evita ambigüedades y reduce el riesgo de introducir data leakage. En esta versión, los targets también se calculan por segmento continuo, por lo que las últimas velas antes de un gap no utilizan la primera vela posterior al hueco como si fuera una observación futura regular.

In [5]:
# ============================================================
# Support and resistance features
# Esta celda genera variables relacionadas con soportes y resistencias utilizando únicamente información histórica.
# El objetivo es aproximar conceptos clásicos del análisis técnico mediante métricas cuantitativas utilizables por modelos de machine learning.
# Las ventanas de soporte y resistencia se calculan por segmento para no mezclar rangos separados por gaps temporales.
# ============================================================

# Rolling range position
range_window = 288  # 24 hours of 5m candles

rolling_low = df.groupby("segment_id")["low"].transform(
    lambda s: s.rolling(window=range_window).min()
)
rolling_high = df.groupby("segment_id")["high"].transform(
    lambda s: s.rolling(window=range_window).max()
)
rolling_range = rolling_high - rolling_low

# Relative position of current price inside recent range
# 0 = near recent low, 1 = near recent high

df["price_position_in_recent_range"] = np.where(
    rolling_range != 0,
    (df["close"] - rolling_low) / rolling_range,
    np.nan
)

# Dynamic support and resistance levels
support_window = 288

# Recent support/resistance approximations
df["recent_support"] = df.groupby("segment_id")["low"].transform(
    lambda s: s.rolling(window=support_window).min()
)
df["recent_resistance"] = df.groupby("segment_id")["high"].transform(
    lambda s: s.rolling(window=support_window).max()
)

# Distance to support/resistance
# Normalized by price to make values comparable across market regimes

df["dist_to_nearest_support"] = (
    (df["close"] - df["recent_support"])
    / df["close"]
)

df["dist_to_nearest_resistance"] = (
    (df["recent_resistance"] - df["close"])
    / df["close"]
)

# Touch counters
# Counts how many times price approached the support/resistance area recently

tolerance = 0.003  # 0.3%

near_support_condition = (
    (df["low"] - df["recent_support"]).abs() / df["close"] < tolerance
)
near_resistance_condition = (
    (df["high"] - df["recent_resistance"]).abs() / df["close"] < tolerance
)

df["near_support"] = np.where(
    df["recent_support"].notna(),
    near_support_condition.astype(float),
    np.nan
)

df["near_resistance"] = np.where(
    df["recent_resistance"].notna(),
    near_resistance_condition.astype(float),
    np.nan
)

# Strength estimation via rolling touch counts
strength_window = 288

df["support_strength"] = df.groupby("segment_id")["near_support"].transform(
    lambda s: s.rolling(window=strength_window).sum()
)

df["resistance_strength"] = df.groupby("segment_id")["near_resistance"].transform(
    lambda s: s.rolling(window=strength_window).sum()
)

# Combined touch count feature
df["touch_count_near_level"] = (
    df["support_strength"]
    + df["resistance_strength"]
)

print("Gap-aware support/resistance features created")

support_cols = [
    "price_position_in_recent_range",
    "dist_to_nearest_support",
    "dist_to_nearest_resistance",
    "support_strength",
    "resistance_strength",
    "touch_count_near_level"
]

print(df[support_cols].describe())

Gap-aware support/resistance features created
       price_position_in_recent_range  dist_to_nearest_support  \
count                   717956.000000            717956.000000   
mean                         0.498585                 0.039175   
std                          0.256457                 0.047523   
min                          0.000000                 0.000000   
25%                          0.283820                 0.014252   
50%                          0.494382                 0.026175   
75%                          0.711651                 0.046755   
max                          1.000000                 0.905706   

       dist_to_nearest_resistance  support_strength  resistance_strength  \
count               717956.000000     712503.000000        712503.000000   
mean                     0.040189         12.522145            15.029252   
std                      0.051215         18.056414            21.622232   
min                      0.000000          0.000000    

Estas variables intentan traducir conceptos subjetivos del análisis técnico clásico a señales cuantificables. En lugar de asumir manualmente dónde existen soportes o resistencias, el sistema utiliza ventanas móviles y densidad de toques recientes para estimar zonas relevantes del precio.

Aunque este enfoque sigue siendo relativamente simple frente a técnicas más avanzadas de clustering o detección estructural de niveles, permite incorporar contexto de mercado adicional que no capturan indicadores tradicionales como RSI o MACD. Además, todas las variables se calculan exclusivamente con información histórica y dentro de segmentos temporales continuos, manteniendo la coherencia necesaria para el entrenamiento de modelos predictivos.

In [6]:
# ============================================================
# Clean and save processed dataset
# Elimina las filas con valores nulos generados por ventanas móviles, EMAs iniciales, desplazamientos futuros y cortes de segmento. Después guarda el dataset final en la carpeta data/processed.
# ============================================================

print("Shape before dropna:", df.shape)
print("Missing values before dropna:", df.isna().sum().sum())

rows_before_cleaning = df.shape[0]
metadata_cols = ["time_diff", "gap_flag", "segment_id"]
target_label_cols = [f"up_{h}" for h in horizons]

df_features = (
    df.dropna()
    .drop(columns=metadata_cols)
    .reset_index(drop=True)
)

df_features[target_label_cols] = df_features[target_label_cols].astype(int)
rows_removed = rows_before_cleaning - df_features.shape[0]

print("Shape after dropna:", df_features.shape)
print("Missing values after dropna:", df_features.isna().sum().sum())
print("Rows removed by rolling windows, labels and gap-aware processing:", rows_removed)

# Informative check: remaining jumps only mark transitions between independent continuous segments after cleaning.
remaining_time_diff = df_features["open_time"].diff()
remaining_non_regular_jumps = remaining_time_diff.ne(EXPECTED_FREQ).sum() - 1
remaining_non_regular_jumps = max(int(remaining_non_regular_jumps), 0)

print("Remaining non-regular jumps after cleaning:", remaining_non_regular_jumps)
print("These jumps are expected between cleaned independent segments and were not crossed during feature calculation.")

df_features.to_csv(OUTPUT_PATH, index=False)

print(f"Saved processed dataset to: {OUTPUT_PATH}")

display(df_features.head())
display(df_features.tail())

Shape before dropna: (723409, 57)
Missing values before dropna: 75698
Shape after dropna: (712257, 54)
Missing values after dropna: 0
Rows removed by rolling windows, labels and gap-aware processing: 11152
Remaining non-regular jumps after cleaning: 25
These jumps are expected between cleaned independent segments and were not crossed during feature calculation.
Saved processed dataset to: ../data/processed\DOGEUSDT_5m_binance_2017_2026_features.csv


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,price_position_in_recent_range,recent_support,recent_resistance,dist_to_nearest_support,dist_to_nearest_resistance,near_support,near_resistance,support_strength,resistance_strength,touch_count_near_level
0,2019-07-07 11:50:00,0.003496,0.003503,0.003492,0.003492,145316.0,2019-07-07 11:54:59.999,508.053883,14,37501.0,...,0.402795,0.003365,0.00368,0.036312,0.053837,0.0,0.0,30.0,0.0,30.0
1,2019-07-07 11:55:00,0.003494,0.003500,0.003492,0.003500,1727089.0,2019-07-07 11:59:59.999,6040.557436,21,1017415.0,...,0.429479,0.003365,0.00368,0.038624,0.051308,0.0,0.0,30.0,0.0,30.0
2,2019-07-07 12:00:00,0.003497,0.003505,0.003491,0.003505,2644825.0,2019-07-07 12:04:59.999,9268.834500,26,811589.0,...,0.443456,0.003365,0.00368,0.039831,0.049989,0.0,0.0,30.0,0.0,30.0
3,2019-07-07 12:05:00,0.003505,0.003505,0.003487,0.003498,1335881.0,2019-07-07 12:09:59.999,4672.287479,35,1267575.0,...,0.422490,0.003365,0.00368,0.038020,0.051970,0.0,0.0,30.0,0.0,30.0
4,2019-07-07 12:10:00,0.003498,0.003516,0.003498,0.003505,5442392.0,2019-07-07 12:14:59.999,19046.081219,64,5027347.0,...,0.444091,0.003365,0.00368,0.039886,0.049929,0.0,0.0,30.0,0.0,30.0


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,price_position_in_recent_range,recent_support,recent_resistance,dist_to_nearest_support,dist_to_nearest_resistance,near_support,near_resistance,support_strength,resistance_strength,touch_count_near_level
712252,2026-05-23 09:50:00,0.09950,0.09952,0.09943,0.09946,1679336.0,2026-05-23 09:54:59.999,167050.11271,1127,783613.0,...,0.221987,0.09736,0.10682,0.021114,0.074000,0.0,0.0,34.0,18.0,52.0
712253,2026-05-23 09:55:00,0.09947,0.09948,0.09938,0.09940,1066829.0,2026-05-23 09:59:59.999,106080.10005,780,384552.0,...,0.215645,0.09736,0.10682,0.020523,0.074648,0.0,0.0,34.0,18.0,52.0
712254,2026-05-23 10:00:00,0.09941,0.09947,0.09937,0.09942,815385.0,2026-05-23 10:04:59.999,81075.92691,997,222029.0,...,0.217759,0.09736,0.10682,0.020720,0.074432,0.0,0.0,34.0,18.0,52.0
712255,2026-05-23 10:05:00,0.09942,0.09948,0.09937,0.09946,3255114.0,2026-05-23 10:09:59.999,323681.19633,1232,2459145.0,...,0.221987,0.09736,0.10682,0.021114,0.074000,0.0,0.0,34.0,18.0,52.0
712256,2026-05-23 10:10:00,0.09945,0.09955,0.09945,0.09955,312988.0,2026-05-23 10:14:59.999,31143.08415,922,176706.0,...,0.231501,0.09736,0.10682,0.021999,0.073029,0.0,0.0,34.0,18.0,52.0


La limpieza elimina las filas afectadas por ventanas móviles iniciales, indicadores que requieren histórico suficiente, etiquetas futuras no disponibles y zonas cercanas a discontinuidades temporales. El dataset final queda sin valores nulos y preparado para las siguientes fases del proyecto: análisis exploratorio, entrenamiento de modelos y backtesting.

La presencia de algunos saltos temporales entre segmentos en el dataset final no implica que los indicadores hayan cruzado dichos huecos. Cada segmento se ha procesado de forma independiente, por lo que las features y targets se mantienen temporalmente coherentes dentro de cada tramo continuo.

# Conclusión

Este notebook convierte el histórico bruto de DOGE/USDT en un dataset preparado para machine learning supervisado. El proceso añade indicadores técnicos, variables derivadas y targets futuros claramente diferenciados, manteniendo la lógica temporal necesaria en problemas de series temporales financieras.

El punto más importante no es sólo haber generado más columnas, sino haber separado correctamente las variables explicativas, basadas en información pasada, de las etiquetas que representan el comportamiento futuro. Esta distinción será clave para que los resultados de los modelos y de los backtests posteriores sean interpretables y no estén contaminados por información que no existiría en un escenario real de trading.

Adicionalmente, el tratamiento de gaps temporales mediante segmentos continuos evita que retornos, rolling windows o labels futuros atraviesen discontinuidades en la serie. Esta decisión permite conservar los datos reales disponibles sin introducir velas artificiales y refuerza la consistencia metodológica del pipeline.